In [139]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
# standard scaler
from sklearn.preprocessing import StandardScaler, LabelEncoder
from datetime import timedelta


In [ ]:
class StateDataset(torch.utils.data.Dataset):
    def __init__(self, df, feature_cols, target_col, lookback, forecast_window,start_test_date, split=None, skip_nan_windows=True):
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.lookback = lookback
        self.forecast_window = forecast_window
        self.start_test_date = start_test_date
        self.split = split
        self.skip_nan_windows = skip_nan_windows
        self.Xdata = []
        self.Ydata = []
        self.state_data = []
        # Group by state and create sequences
        if start_test_date is not None:
            start_test_date = pd.to_datetime(start_test_date)
        for state, group in df.groupby('state'):
            group = group.sort_values('date').reset_index(drop=True)
            features = group[feature_cols].values
            targets = group[target_col].values
            dates = group['date'].values
            for i in range(len(group) - lookback - forecast_window+1):
                y_start_date = pd.to_datetime(dates[i + lookback])
                if split == 'train' and y_start_date >= start_test_date:
                    continue   
                if split == 'test' and y_start_date < start_test_date:
                    continue

                X_window = features[i:i + lookback]
                y_window = targets[i + lookback: i + lookback + forecast_window]

                if skip_nan_windows and (np.isnan(X_window).any() or np.isnan(y_window).any()):
                    continue

                self.Xdata.append(X_window)
                self.Ydata.append(y_window)
                self.state_data.append(state)
        self.Xdata = torch.FloatTensor(np.array(self.Xdata))   
        self.Ydata = torch.FloatTensor(np.array(self.Ydata))
        self.state_data = torch.LongTensor(np.array(self.state_data))   

    def __len__(self):
        return len(self.Xdata)

    def __getitem__(self, idx):
        return self.Xdata[idx], self.state_data[idx], self.Ydata[idx]

In [ ]:
def preprocessing_pipeline(df,target_col,start_test_date):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    start_test_date = pd.to_datetime(start_test_date)
    state_encoder = LabelEncoder()
    df['state'] = state_encoder.fit_transform(df['state'])
    # create sine and cosine columns based on weeks and drop week column which is already present
    df['sine_week'] = np.sin(2 * np.pi * df['week'] / 52)
    df['cosine_week'] = np.cos(2 * np.pi * df['week'] / 52)
    df.drop(columns=['week'], inplace=True)
    # create sine and cosine columns based on month and drop month column which is already present
    df['sine_month'] = np.sin(2 * np.pi * df['month']/ 12) 
    df['cosine_month'] = np.cos(2 * np.pi * df['month']/ 12) 
    df.drop(columns=['month'], inplace=True)
    # same with quarter
    df['sine_quarter'] = np.sin(2 * np.pi * df['quarter'] / 4) 
    df['cosine_quarter'] = np.cos(2 * np.pi * df['quarter'] / 4) 
    df.drop(columns=['quarter'], inplace=True)

    cols_to_skip_scaling = [
    target_col,
    'holiday',
    'state',
    'sine_week',
    'cosine_week',
    'sine_month',
    'cosine_month',
    'sine_quarter',
    'cosine_quarter',
    'date',
    'year',
]
    feature_cols = [col for col in df.columns if col not in ['date', 'state', target_col]]

    scale_cols = [col for col in feature_cols if col not in cols_to_skip_scaling]

    df[scale_cols] = df[scale_cols].astype(float)

    df_train_rows = df[df["date"] < start_test_date].copy()
    df_test_rows = df[df["date"] >= start_test_date].copy()

    scaler = StandardScaler()
    scaler.fit(df_train_rows[scale_cols])
    df.loc[:, scale_cols] = scaler.transform(df.loc[:, scale_cols])
    df_train_rows = df[df["date"] < start_test_date].copy()
    df_test_rows = df[df["date"] >= start_test_date].copy()
    return df_train_rows, df_test_rows

In [ ]:
def create_dataloaders(df, start_test_date, target_col='cases_per_100k', lookback=52, forecast_window=2, batch_size=32):
    df_train_rows, df_test_rows = preprocessing_pipeline(df, target_col, start_test_date)
    df_processed = pd.concat([df_train_rows, df_test_rows], ignore_index=True)
    feature_cols = [col for col in df_processed.columns if col not in ['date', 'state', target_col]]
    train_dataset = StateDataset(df_processed, feature_cols, target_col, lookback, forecast_window, start_test_date, split='train')
    test_dataset = StateDataset(df_processed, feature_cols, target_col, lookback, forecast_window, start_test_date, split='test')
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader, train_dataset, test_dataset, feature_cols

### Rows, samples, and batches

In this dataset, one raw dataframe row is one weekly observation for one state. A model sample is larger than one row: it is one sliding time window made from several rows.

With `lookback = 52` and `forecast_window = 2`, each sample contains:

- `X`: 52 weeks of historical input features
- `y`: the next 2 future values of `cases_per_100k`
- `state`: one encoded state id for that whole 52-week sequence

So a sample is conceptually: use weeks `i` through `i + 51` to predict weeks `i + 52` and `i + 53`.

A batch is a group of samples sent to the model at the same time. With `batch_size = 32`, a full batch contains 32 sliding-window samples.

That is why the dataloader returns shapes like:

- `X_batch`: `[32, 52, 46]` = 32 samples, 52 weeks per sample, 46 features per week
- `state_batch`: `[32]` = one state id per sample
- `y_batch`: `[32, 2]` = 32 samples, each with 2 future target values

The number of samples is smaller than the number of raw rows because each sample requires `lookback + forecast_window` rows. For a state with `N` weekly rows, the number of possible windows is `N - lookback - forecast_window + 1`. The final batch can have fewer than 32 samples if the total sample count is not divisible by `batch_size`.



print("Train samples:", len(df_train))
print("Test samples:", len(df_test))

for batch_idx, (X_batch, state_batch, y_batch) in enumerate(train_loader):
    print(f"\nTrain batch {batch_idx}")
    print("X_batch shape:", X_batch.shape)
    print("state_batch shape:", state_batch.shape)
    print("y_batch shape:", y_batch.shape)

    print("X_batch example:", X_batch[0])
    print("state example:", state_batch[0])
    print("y_batch example:", y_batch[0])

Train samples: 3356
Test samples: 432

Train batch 0
X_batch shape: torch.Size([32, 52, 46])
state_batch shape: torch.Size([32])
y_batch shape: torch.Size([32, 2])
X_batch example: tensor([[1.0000e+00, 0.0000e+00, 1.7760e+00,  ..., 5.0000e-01, 1.0000e+00,
         6.1232e-17],
        [1.0000e+00, 1.0000e+00, 7.0667e-01,  ..., 5.0000e-01, 1.0000e+00,
         6.1232e-17],
        [1.0000e+00, 0.0000e+00, 7.9578e-01,  ..., 5.0000e-01, 1.0000e+00,
         6.1232e-17],
        ...,
        [2.0000e+00, 0.0000e+00, 2.8899e+00,  ..., 8.6603e-01, 1.0000e+00,
         6.1232e-17],
        [2.0000e+00, 0.0000e+00, 1.5978e+00,  ..., 5.0000e-01, 1.0000e+00,
         6.1232e-17],
        [2.0000e+00, 0.0000e+00, 2.3107e+00,  ..., 5.0000e-01, 1.0000e+00,
         6.1232e-17]])
state example: tensor(1)
y_batch example: tensor([ 8.1067, 10.1333])

Train batch 1
X_batch shape: torch.Size([32, 52, 46])
...
        [ 2.0000e+00,  0.0000e+00,  1.8800e-01,  ...,  8.6603e-01,
         -2.4493e-16,  1.0000e+00]])
state example: tensor(6)
y_batch example: tensor([3.4673, 5.1841])